# Automated B2B Sales Prospecting (AutoGen + Gradio)

Intent agent -> web discovery (first real company + public contact email, then stop) -> Market Analyst -> Copywriter <-> Compliance -> checkbox approval -> SendGrid to that address only. Respect site terms when fetching pages.

### Import relevant libraries

In [ ]:
import asyncio
import io
import os
import re
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import urljoin, urlparse

import gradio as gr
import httpx
import sendgrid
from ddgs import DDGS
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from rich.console import Console
from sendgrid.helpers.mail import Content, Email, Mail, To

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient

load_dotenv(override=True)

_langchain_llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0)

console = Console(force_terminal=True, width=80)
_trace_console: Optional[Console] = None


def _active_console() -> Console:
    return _trace_console if _trace_console is not None else console

### Set up environment

In [ ]:
SENDER_EMAIL = "answers@ailysis.io"

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
SENDGRID_API_KEY = os.environ.get("SENDGRID_API_KEY")

print("Environment ready.")

### Business Tools

In [ ]:
def _send_marketing_email_impl(subject: str, html_content: str, recipients: List[str]) -> str:
    """Send via SendGrid."""
    _active_console().print(
        f"[bold green]Dispatching email to {len(recipients)} recipient(s)...[/bold green]"
    )
    api_key = os.environ.get("SENDGRID_API_KEY")
    if not api_key:
        return "Skipped send: SENDGRID_API_KEY is not set (dry run)."

    try:
        sg = sendgrid.SendGridAPIClient(api_key=api_key)
        from_email = Email(SENDER_EMAIL)
        for email_addr in recipients:
            to_email = To(email_addr)
            content = Content("text/html", html_content)
            mail = Mail(from_email, to_email, subject, content)
            response = sg.client.mail.send.post(request_body=mail.get())
            if response.status_code not in (200, 201, 202):
                return f"SendGrid error {response.status_code}: {response.body}"
        return f"Successfully sent to {len(recipients)} recipients."
    except Exception as e:
        return f"Error: {str(e)}"

### AutoGen pipeline

> Intent_Extractor (structured) -> web + contact scrape (first verified email wins) -> Market_Analyst -> RoundRobinGroupChat (Sales_Copywriter <-> Compliance_Tone_Reviewer, APPROVED). Sending happens only after approval.

In [ ]:
AUTOGEN_MODEL = "gpt-4o-mini"

APP_PRODUCT = (
    "Mobile app: automates receipt scanning, extracts line-item data, supports financial analysis, "
    "budgets, AI-powered chat, and data exports."
)

EMAIL_RE = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
_BAD_EMAIL_HOST = (
    "example.com", "test.com", "domain.com", "yoursite.com", "wix.com", "schema.org",
    "sentry.io", "w3.org", "github.com", "localhost",
)
_BAD_URL_HOST = (
    "facebook.com", "instagram.com", "tiktok.com", "linkedin.com", "twitter.com",
    "youtube.com", "pinterest.com", "reddit.com",
)


class ProspectingIntent(BaseModel):
    summary: str = Field(description="One-sentence restatement of the user's prospecting goal.")
    target_company_profile: str = Field(description="Ideal companies to find: industry, size, geo, keywords.")
    inferred_segment: str = Field(description="Accounting Firm, Small Business, or Mixed.")
    pain_hypothesis: str = Field(description="Pain hypotheses matching receipt/bookkeeping automation.")
    suggested_search_queries: List[str] = Field(
        description="3–6 DuckDuckGo-style queries to find real businesses likely to publish a contact email.",
        min_length=3,
        max_length=6,
    )


class EmailDraft(BaseModel):
    subject: str = Field(description="Email subject line.")
    html_body: str = Field(description="HTML body for the outreach email.")


def _title_to_company(title: str) -> str:
    t = title or ""
    for sep in ("|", " · ", " – ", " - ", " — "):
        if sep in t:
            t = t.split(sep)[0]
    return (t or "Unknown company").strip()[:120]


def _is_plausible_contact_email(addr: str) -> bool:
    addr = addr.strip()
    if "@" not in addr or len(addr) < 6:
        return False
    local, _, domain = addr.rpartition("@")
    if len(local) < 1 or "." not in domain:
        return False
    dl = domain.lower()
    if dl.endswith((".png", ".jpg", ".jpeg", ".gif", ".webp", ".css", ".js")):
        return False
    return not any(b in dl for b in _BAD_EMAIL_HOST)


def _pick_preferred_email(candidates: List[str]) -> Optional[str]:
    cleaned: List[str] = []
    for c in candidates:
        if not _is_plausible_contact_email(c):
            continue
        cleaned.append(c.strip().strip(".,);]'\" "))
    if not cleaned:
        return None
    cleaned = list(dict.fromkeys(c for c in cleaned if _is_plausible_contact_email(c)))
    priority = ("contact@", "info@", "hello@", "office@", "admin@", "sales@")
    for p in priority:
        for e in cleaned:
            if e.lower().startswith(p):
                return e
    return cleaned[0]


def _extract_emails_from_text(text: str) -> List[str]:
    found: List[str] = []
    for m in re.finditer(r'mailto:([^"\'>\s&]+)', text, re.I):
        addr = m.group(1).split("?")[0].strip()
        if _is_plausible_contact_email(addr):
            found.append(addr)
    for m in EMAIL_RE.finditer(text):
        if _is_plausible_contact_email(m.group(0)):
            found.append(m.group(0))
    return found


def _url_ok(url: str) -> bool:
    try:
        p = urlparse(url)
        if p.scheme not in ("http", "https") or not p.netloc:
            return False
        h = p.netloc.lower()
        return not any(b in h for b in _BAD_URL_HOST)
    except Exception:
        return False


def _contact_url_variants(seed_url: str) -> List[str]:
    if not _url_ok(seed_url):
        return []
    p = urlparse(seed_url)
    base = f"{p.scheme}://{p.netloc}"
    paths = [
        seed_url,
        urljoin(base + "/", "contact"),
        urljoin(base + "/", "contact-us"),
        urljoin(base + "/", "about"),
        urljoin(base + "/", "about-us"),
    ]
    return list(dict.fromkeys(paths))


async def _fetch_text(client: httpx.AsyncClient, url: str) -> str:
    try:
        r = await client.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0 (compatible; B2BProspector/1.0; +https://example.invalid)"
            },
        )
        if r.status_code >= 400:
            return ""
        return r.text[:600_000]
    except Exception:
        return ""


def _ddgs_text_search(query: str, max_results: int = 8) -> List[dict]:
    try:
        return list(DDGS().text(query, max_results=max_results))
    except Exception as e:
        _active_console().print(f"[yellow]DDGS: {e}[/yellow]")
        return []


async def extract_intent_from_query(
    user_query: str, model_client: OpenAIChatCompletionClient
) -> ProspectingIntent:
    cancellation_token = CancellationToken()
    intent_agent = AssistantAgent(
        name="Intent_Extractor",
        model_client=model_client,
        system_message=(
            "Parse the user's prospecting request. Output structured fields only via schema. "
            "Search queries must be short English strings suitable for public web search to find REAL small firms "
            "or accounting practices that typically list info@, contact@, or hello@ on a /contact page. "
            "Include the product context: receipt scanning, bookkeeping automation, SMB/accounting."
        ),
        output_content_type=ProspectingIntent,
    )
    resp = await intent_agent.on_messages(
        [TextMessage(content=user_query, source="user")],
        cancellation_token=cancellation_token,
    )
    content = resp.chat_message.content
    if isinstance(content, ProspectingIntent):
        return content
    raise RuntimeError("Intent extraction did not return ProspectingIntent")


async def discover_first_lead_with_email(
    intent: ProspectingIntent,
    client: httpx.AsyncClient,
) -> Optional[Dict[str, Any]]:
    queries: List[str] = list(dict.fromkeys(intent.suggested_search_queries))
    extra = [
        f"{intent.target_company_profile} contact email",
        f"{intent.target_company_profile} bookkeeping firm contact",
    ]
    for q in extra:
        if q not in queries:
            queries.append(q)

    for q in queries:
        results = await asyncio.to_thread(_ddgs_text_search, q, 8)
        for row in results:
            url = row.get("href") or ""
            title = row.get("title") or ""
            body = row.get("body") or ""
            if not _url_ok(url):
                continue

            company = _title_to_company(title)
            snippet_emails = _extract_emails_from_text(body)
            picked = _pick_preferred_email(snippet_emails)
            if picked:
                return {
                    "entity_name": company,
                    "segment": intent.inferred_segment,
                    "pain_point": intent.pain_hypothesis,
                    "trigger": intent.summary,
                    "lead_email": picked,
                    "contact_url": url,
                    "source_query": q,
                }

            for page_url in _contact_url_variants(url):
                html = await _fetch_text(client, page_url)
                emails = _extract_emails_from_text(html)
                picked = _pick_preferred_email(emails)
                if picked:
                    return {
                        "entity_name": company,
                        "segment": intent.inferred_segment,
                        "pain_point": intent.pain_hypothesis,
                        "trigger": intent.summary,
                        "lead_email": picked,
                        "contact_url": page_url,
                        "source_query": q,
                    }

    return None


def _lead_brief(lead: dict, user_query: str) -> str:
    lines = [
        f"**Prospect:** {lead['entity_name']}",
        f"**Segment:** {lead['segment']}",
        f"**Pain point:** {lead['pain_point']}",
        f"**Trigger / timing:** {lead['trigger']}",
        f"**Verified contact email (send only after human approval in UI):** {lead['lead_email']}",
        f"**Contact page sourced:** {lead.get('contact_url', '')}",
        f"**Product:** {APP_PRODUCT}",
        f"**Original user request:** {user_query.strip()}",
    ]
    return "\n".join(lines)


def _format_messages_trace(messages) -> str:
    chunks: List[str] = []
    for m in messages:
        src = getattr(m, "source", "?")
        content = getattr(m, "content", "")
        text = content if isinstance(content, str) else str(content)
        if len(text) > 12000:
            text = text[:12000] + "\n… [truncated]"
        chunks.append(f"**{src}**\n\n{text}")
    return "\n\n---\n\n".join(chunks)


def _extract_final_email(text: str) -> Optional[EmailDraft]:
    subj = re.search(r"(?im)^SUBJECT:\s*(.+)$", text)
    body = re.search(r"(?is)^BODY_HTML:\s*(.+?)(?=^SUBJECT:|^APPROVED\s*$|\Z)", text)
    if subj and body:
        return EmailDraft(subject=subj.group(1).strip(), html_body=body.group(1).strip())
    try:
        extractor = _langchain_llm.with_structured_output(EmailDraft)
        return extractor.invoke(
            "Extract the final approved outreach email (subject + html_body) from:\n\n" + text[:14000]
        )
    except Exception:
        return None


async def run_b2b_pipeline_for_lead(
    lead: dict,
    user_query: str,
    model_client: OpenAIChatCompletionClient,
) -> Tuple[str, str, Optional[EmailDraft]]:
    """Returns (analyst strategy, full conversation markdown, final draft or None)."""
    cancellation_token = CancellationToken()
    brief = _lead_brief(lead, user_query)

    market_analyst = AssistantAgent(
        name="Market_Analyst",
        model_client=model_client,
        system_message=(
            "You are a B2B market analyst. From segment and pain point, choose the most relevant product angles "
            "(e.g. bulk export, multi-entity workflows, audit-friendly exports for Accounting Firms; "
            "budgeting, fast capture, AI chat for Small Business). Output a short bullet strategy only—no email."
        ),
    )
    ra = await market_analyst.on_messages(
        [TextMessage(content=brief, source="user")],
        cancellation_token=cancellation_token,
    )
    strategy = str(ra.chat_message.content)

    sales_copywriter = AssistantAgent(
        name="Sales_Copywriter",
        model_client=model_client,
        system_message=(
            "You write personalized B2B outreach. Open with the trigger and pain; tie features to outcomes. "
            "The email will be sent ONLY after a human approves in the app—do not claim it was already sent. "
            "Use this exact format: a line SUBJECT: ... then a line BODY_HTML: then the HTML body. "
            "If Compliance rejects the draft, revise with the same format. Avoid hype and generic claims."
        ),
    )
    compliance = AssistantAgent(
        name="Compliance_Tone_Reviewer",
        model_client=model_client,
        system_message=(
            "You are Compliance & Tone Reviewer. Accounting Firm prospects: professional, precise, understated. "
            "Small Business: clear, friendly, non-jargony. Reject copy that is generic, pushy, or 'salesy'. "
            "When acceptable, include APPROVED alone on its own line and preserve final SUBJECT:/BODY_HTML: blocks. "
            "Do not mention sending tools—the user approves send separately in the UI."
        ),
    )

    team_task = (
        f"{brief}\n\n## Market_Analyst strategy\n{strategy}\n\n"
        "Sales_Copywriter: produce the outreach email (SUBJECT:/BODY_HTML:). "
        "Compliance_Tone_Reviewer: review and iterate until you can output APPROVED with final copy."
    )
    team = RoundRobinGroupChat(
        [sales_copywriter, compliance],
        termination_condition=TextMentionTermination("APPROVED"),
        max_turns=14,
    )
    team_result = await team.run(task=team_task)

    analyst_trace = _format_messages_trace([TextMessage(content=brief, source="user"), ra.chat_message])
    team_trace = _format_messages_trace(team_result.messages)
    full_trace = f"### Analyst phase\n\n{analyst_trace}\n\n### Copywriter ↔ Compliance\n\n{team_trace}"

    final_draft: Optional[EmailDraft] = None
    for msg in reversed(team_result.messages):
        content = str(getattr(msg, "content", "") or "")
        if "APPROVED" in content and "BODY_HTML:" in content:
            final_draft = _extract_final_email(content)
            if final_draft:
                break
    if final_draft is None and team_result.messages:
        final_draft = _extract_final_email(str(getattr(team_result.messages[-1], "content", "")))

    return strategy, full_trace, final_draft


### Streaming logic

In [ ]:
def _build_gradio_markdown(
    ansi_block: str,
    intent_md: str,
    discovery_md: str,
    history_md: str,
    emails_md: str,
) -> str:
    parts = [ansi_block]
    if intent_md.strip():
        parts.append("## Extracted intent\n\n" + intent_md)
    if discovery_md.strip():
        parts.append("## Lead discovery\n\n" + discovery_md)
    if history_md.strip():
        parts.append("## AutoGen conversation history\n\n" + history_md)
    if emails_md.strip():
        parts.append("## Draft email (preview)\n\n" + emails_md)
    return "\n\n".join(parts)


async def run_discovery_stream(user_query: str):
    """Intent → first real email found → draft agents. SendGrid only after UI approval (separate button)."""
    global _trace_console
    buf = io.StringIO()
    capture_console = Console(file=buf, force_terminal=True, width=100)
    _trace_console = capture_console

    intent_md = ""
    discovery_md = ""
    history_md = ""
    emails_md = ""
    state_payload: Optional[Dict[str, Any]] = None

    def emit() -> str:
        ansi = f"```ansi\n{buf.getvalue()}\n```"
        return _build_gradio_markdown(ansi, intent_md, discovery_md, history_md, emails_md)

    model_client: Optional[OpenAIChatCompletionClient] = None
    try:
        model_client = OpenAIChatCompletionClient(
            model=AUTOGEN_MODEL,
            api_key=os.environ.get("OPENAI_API_KEY"),
        )
        capture_console.print(
            "[bold bright_cyan]Intent → search → first contact email → draft[/bold bright_cyan]"
        )
        yield emit(), state_payload, ""

        capture_console.print("[yellow]Extracting intent...[/yellow]")
        yield emit(), state_payload, ""
        intent = await extract_intent_from_query(user_query, model_client)
        intent_md = intent.model_dump_json(indent=2)
        capture_console.print("[green]Intent structured.[/green]")
        yield emit(), state_payload, ""

        capture_console.print("[yellow]Searching for one company with a contact email (stops at first hit)...[/yellow]")
        yield emit(), state_payload, ""

        async with httpx.AsyncClient(timeout=httpx.Timeout(20.0), follow_redirects=True) as http_client:
            lead = await discover_first_lead_with_email(intent, http_client)

        if not lead:
            discovery_md = "_No lead with a discoverable email. Try different wording or geography._"
            capture_console.print("[red]Stopped: no email found.[/red]")
            yield emit(), state_payload, ""
            return

        discovery_md = (
            f"**Company:** {lead['entity_name']}\n\n"
            f"**Email:** `{lead['lead_email']}`\n\n"
            f"**Sourced from:** {lead.get('contact_url', '')}\n\n"
            f"**Search query:** {lead.get('source_query', '')}"
        )
        capture_console.print(f"[bold green]Found: {lead['lead_email']}[/bold green]")
        yield emit(), state_payload, ""

        capture_console.print("[magenta]Drafting with AutoGen team...[/magenta]")
        yield emit(), state_payload, ""

        _strategy, trace, draft = await run_b2b_pipeline_for_lead(lead, user_query, model_client)
        history_md = trace
        if draft:
            emails_md = (
                f"**To:** `{lead['lead_email']}`\n\n"
                f"**Subject:** {draft.subject}\n\n{draft.html_body}"
            )
            state_payload = {
                "to": lead["lead_email"],
                "subject": draft.subject,
                "html": draft.html_body,
            }
            capture_console.print("[green]Draft ready — approve below, then SendGrid.[/green]")
        else:
            emails_md = "_Agents did not produce a parseable draft._"
            capture_console.print("[red]Draft parse failed.[/red]")
        yield emit(), state_payload, ""

    except Exception as e:
        capture_console.print(f"\n[bold red]Critical system error:[/bold red] {e!s}")
        yield emit(), state_payload, ""
    finally:
        if model_client is not None:
            await model_client.close()
        _trace_console = None


def send_approved_email(
    approved: bool, payload: Optional[Dict[str, Any]]
) -> Tuple[str, Optional[Dict[str, Any]]]:
    """SendGrid only if the user checks approve."""
    if not approved:
        return (
            "### Not sent\nSelect **I approve…** and click **Send with SendGrid** to deliver.",
            payload,
        )
    if not payload or not payload.get("to"):
        return "### Not sent\nRun discovery first.", payload
    status = _send_marketing_email_impl(payload["subject"], payload["html"], [payload["to"]])
    if status.startswith("Successfully"):
        return f"### {status}\nDelivered to `{payload['to']}`.", None
    return f"### Send result\n{status}", payload


### UI with execution trace

In [ ]:
def launch_ui():
    dark_mode_js = """
    function() {
        document.documentElement.classList.add('dark');
        document.body.classList.add('dark');
    }
    """

    dark_theme = gr.themes.Default(
        primary_hue=gr.themes.colors.blue,
        neutral_hue=gr.themes.colors.slate
    )

    with gr.Blocks(theme=dark_theme, js=dark_mode_js) as demo:
        gr.Markdown(
            "# AI B2B prospecting (AutoGen)\n\n"
            "Describe who you want to reach. The app extracts intent, searches the open web for **one** real company "
            "with a **contact email**, drafts outreach, then lets you **approve** before SendGrid sends."
        )

        draft_state = gr.State(None)

        with gr.Row():
            with gr.Column(scale=1):
                prompt_input = gr.Textbox(
                    label="Your prospecting query",
                    placeholder="e.g. Independent CPAs in Denver that might want receipt automation for clients",
                    lines=6,
                )
                discover_btn = gr.Button("Find lead & draft email", variant="primary")
                approve_cb = gr.Checkbox(
                    label="I approve sending this exact draft to the discovered email",
                    value=False,
                )
                send_btn = gr.Button("Send with SendGrid", variant="secondary")

            with gr.Column(scale=2):
                trace_output = gr.Markdown(
                    label="Run log, intent, discovery, agents, draft",
                    sanitize_html=False,
                )
                send_status = gr.Markdown(label="Send status")

        discover_btn.click(
            fn=run_discovery_stream,
            inputs=prompt_input,
            outputs=[trace_output, draft_state, send_status],
            show_progress="minimal",
        )
        send_btn.click(
            fn=send_approved_email,
            inputs=[approve_cb, draft_state],
            outputs=[send_status, draft_state],
        )

    demo.launch(share=True)


if __name__ == "__main__":
    launch_ui()
